# 05 — 模型注册表管理

**前置：** 已完成 `end_to_end_training_pipeline.ipynb`（已注册模型到 SQLite）。

**职责：**
- 离线列出 + 对比已注册的所有模型
- 并排 equity curve 可视化
- 标记 STAGING → PRODUCTION 晋级决策
- IC 衰减分析（按月/季度切分 IC，判断是否需要重训）

**不依赖 API server**，完全离线运行。

## ⚙️ 读取 session 配置

In [ ]:
import sys, json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

cfg_path = ROOT / 'data' / 'session_config.json'
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    MARKET, SYMBOLS, BENCHMARK = cfg['market'], cfg['symbols'], cfg['benchmark']
    TEST_START, TEST_END = cfg['test_start'], cfg['test_end']
    print(f'✅ Loaded session_config.json')
else:
    MARKET, SYMBOLS, BENCHMARK = 'us', ['AAPL','NVDA','MSFT','GOOGL','AMZN','META','TSLA','AVGO','COST','NFLX'], 'QQQ'
    TEST_START, TEST_END = '2025-01-01', '2026-06-18'
    print('⚠️ Using defaults')

from src.assistant.model_registry_index import ModelRegistryIndex
from src.assistant.metadata_db import resolve_metadata_db_path
from src.core.metrics import compute_ic_series
from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init

db_path = resolve_metadata_db_path(ROOT)
reg = ModelRegistryIndex(db_path=db_path)
print(f'Registry: {db_path}')

## 1. 列出所有已注册模型

In [ ]:
entries = reg.list_entries()
if not entries:
    print('⚠️ No models registered yet. Run end_to_end_training_pipeline first.')
else:
    rows = []
    for e in entries:
        bt = e.get('backtest', {}).get('metrics', {})
        rows.append({
            'id': e.get('id','?')[:30],
            'tag': e.get('tag','?'),
            'market': e.get('market','?'),
            'stage': e.get('stage','?'),
            'created': str(e.get('created_at','?'))[:10],
            'sharpe': round(bt.get('sharpe_ratio', float('nan')), 3),
            'total_ret': f"{bt.get('total_return', float('nan')):.2%}" if bt else '?',
            'max_dd': f"{bt.get('max_drawdown', float('nan')):.2%}" if bt else '?',
            'ic': round(bt.get('mean_ic', float('nan')), 4),
        })
    reg_df = pd.DataFrame(rows)
    print(f'Total models: {len(reg_df)}')
    print(reg_df.to_string(index=False))

## 2. 并排 Equity Curve 对比

In [ ]:
# 修改这里选择要对比的模型 tag
COMPARE_TAGS = None  # None = 全部，或 ['us_absret', 'us_excess'] 指定

compare_entries = [e for e in entries if COMPARE_TAGS is None or e.get('tag') in COMPARE_TAGS]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#6366f1','#22c55e','#f59e0b','#ef4444','#8b5cf6','#10b981']

metrics_rows = []
for i, e in enumerate(compare_entries):
    label = f"{e.get('tag','?')} ({e.get('created_at','?')[:10]})"
    color = colors[i % len(colors)]
    bt = e.get('backtest', {}).get('metrics', {})

    # 尝试读取 equity curve 文件
    run_id = e.get('run_id')
    equity_path = None
    if run_id:
        from src.common.paths import ARTIFACTS_DIR
        artifact_dir = ARTIFACTS_DIR / 'artifacts' / run_id
        for p in (artifact_dir / 'equity_curve.csv', artifact_dir / 'portfolio_values.csv'):
            if p.exists():
                equity_path = p; break

    if equity_path:
        eq = pd.read_csv(equity_path)
        # 假设列名 date/value 或 index/portfolio_value
        val_col = [c for c in eq.columns if 'value' in c.lower() or 'portfolio' in c.lower()]
        if val_col:
            vals = eq[val_col[0]].values
            norm = vals / vals[0]
            axes[0].plot(norm, color=color, lw=1.5, label=label, alpha=0.85)
    else:
        axes[0].text(0.05, 0.9 - i*0.08, f'{label}: equity file not found',
                    transform=axes[0].transAxes, fontsize=8, color=color)

    metrics_rows.append({
        'label': label,
        'sharpe': bt.get('sharpe_ratio', float('nan')),
        'total_ret': bt.get('total_return', float('nan')),
        'max_dd': bt.get('max_drawdown', float('nan')),
        'ic': bt.get('mean_ic', float('nan')),
    })

axes[0].axhline(1.0, color='gray', ls=':', lw=0.8)
axes[0].legend(fontsize=7, loc='upper left')
axes[0].set_title('Equity Curve Comparison (Normalised to 1.0)')

# Metrics bar chart
m_df = pd.DataFrame(metrics_rows).set_index('label')
if not m_df.empty and m_df['sharpe'].notna().any():
    m_df['sharpe'].plot(kind='barh', ax=axes[1], color='#6366f1', alpha=0.8)
    axes[1].axvline(0, color='black', lw=0.5)
    axes[1].set_title('Sharpe Ratio Comparison')

plt.tight_layout(); plt.show()
print()
print(m_df.to_string())

## 3. IC 衰减分析 — 按月切分，判断是否需要重训

In [ ]:
# 选一个模型做 IC 衰减分析
TARGET_TAG = None  # None = 自动选最新

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D
from qlib.contrib.data.loader import Alpha158DL

candidates = [e for e in entries if TARGET_TAG is None or e.get('tag') == TARGET_TAG]
if not candidates:
    print('No model found'); raise SystemExit(0)
target = sorted(candidates, key=lambda e: e.get('created_at',''))[-1]
print(f'Analysing: {target["tag"]}  created={target["created_at"][:10]}')

# 加载模型 + 特征
with open(target['path'], 'rb') as f:
    booster = pickle.load(f)

alpha_exprs, _ = Alpha158DL.get_feature_config({'kbar':{},'price':{'windows':[0],'feature':['OPEN','HIGH','LOW','VWAP']},'rolling':{}})
extras = ['$close/Ref($close,5)-1','$close/Ref($close,10)-1','$close/Ref($close,20)-1','Std($close,10)','$volume/Ref($volume,10)-1']
X_test = D.features(SYMBOLS, list(alpha_exprs)+extras, start_time=TEST_START, end_time=TEST_END)
X_test = X_test.fillna(0.0).replace([np.inf,-np.inf], 0.0)
y_raw  = D.features(SYMBOLS, ['Ref($close,-10)/Ref($close,-1)-1'], start_time=TEST_START, end_time=TEST_END)
return_panel = y_raw.rename(columns={y_raw.columns[0]:'return'})

from src.core.signals import generate_scores_panel
score_panel = generate_scores_panel(booster, X_test)
ic_full = compute_ic_series(score_panel, return_panel)
print(f'Full period IC: mean={ic_full["ic_mean"]:.4f}  IR={ic_full["ic_ir"]:.4f}')

In [ ]:
# 按月切分 IC
ic_series = ic_full['ic_series']
monthly_ic = ic_series.groupby(ic_series.index.to_period('M')).mean()
quarterly_ic = ic_series.groupby(ic_series.index.to_period('Q')).mean()

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle(f'IC Decay Analysis — {target["tag"]}', fontweight='bold')

# 月度 IC
colors = ['#22c55e' if v > 0 else '#ef4444' for v in monthly_ic.values]
axes[0,0].bar(range(len(monthly_ic)), monthly_ic.values, color=colors, alpha=0.8)
axes[0,0].set_xticks(range(len(monthly_ic)))
axes[0,0].set_xticklabels([str(p) for p in monthly_ic.index], rotation=45, fontsize=7)
axes[0,0].axhline(0, color='black', lw=0.5)
axes[0,0].axhline(ic_full['ic_mean'], color='blue', ls='--', lw=0.8, label='full mean')
axes[0,0].legend(fontsize=8); axes[0,0].set_title('Monthly IC')

# 季度 IC
colors2 = ['#22c55e' if v > 0 else '#ef4444' for v in quarterly_ic.values]
axes[0,1].bar(range(len(quarterly_ic)), quarterly_ic.values, color=colors2, alpha=0.8)
axes[0,1].set_xticks(range(len(quarterly_ic)))
axes[0,1].set_xticklabels([str(p) for p in quarterly_ic.index], rotation=45, fontsize=8)
axes[0,1].axhline(0, color='black', lw=0.5)
axes[0,1].set_title('Quarterly IC')

# IC rolling 60d
ic_roll = ic_series.rolling(60).mean()
ic_series.plot(ax=axes[1,0], color='#6366f1', lw=0.8, alpha=0.5, label='daily IC')
ic_roll.plot(ax=axes[1,0], color='#f59e0b', lw=2, label='60d rolling mean')
axes[1,0].axhline(0, color='gray', ls=':', lw=0.8)
axes[1,0].legend(fontsize=8); axes[1,0].set_title('IC Rolling Mean (60d)')

# IC 衰减判断
recent_ic = ic_series.tail(60).mean()   # 最近 60 天
early_ic  = ic_series.head(60).mean()   # 最早 60 天
decay = early_ic - recent_ic
retrain_flag = recent_ic < ic_full['ic_mean'] * 0.5 or recent_ic < 0

summary = [
    f'Full period IC mean:  {ic_full["ic_mean"]:.4f}',
    f'Early 60d IC mean:    {early_ic:.4f}',
    f'Recent 60d IC mean:   {recent_ic:.4f}',
    f'IC decay:             {decay:.4f}',
    f'Retrain recommended:  {"⚠️ YES" if retrain_flag else "✅ NO"}',
]
axes[1,1].axis('off')
for j, line in enumerate(summary):
    color = '#ef4444' if 'YES' in line else '#22c55e' if 'NO' in line else '#28251d'
    axes[1,1].text(0.05, 0.85 - j*0.15, line, transform=axes[1,1].transAxes,
                  fontsize=11, fontfamily='monospace', color=color)
axes[1,1].set_title('IC Decay Verdict')

plt.tight_layout(); plt.show()
for line in summary: print(line)

## 4. 晋级决策 — STAGING → PRODUCTION

In [ ]:
# 自动判断：Sharpe > 0.5 且 IC > 0.02 且近期 IC 未大幅衰减
print('=== Promotion Gate ===')
for e in entries:
    bt = e.get('backtest', {}).get('metrics', {})
    sharpe = bt.get('sharpe_ratio', 0)
    ic_mean = bt.get('mean_ic', 0)
    stage = e.get('stage', 'STAGING')
    gate = sharpe > 0.5 and ic_mean > 0.02
    verdict = '✅ PROMOTE' if gate and stage == 'STAGING' else ('🏆 ALREADY PROD' if stage == 'PRODUCTION' else '❌ HOLD')
    print(f'  {e.get("tag","?"):20s} sharpe={sharpe:.2f}  ic={ic_mean:.4f}  stage={stage:10s} → {verdict}')

print()
print('要晋级某个模型，取消下方注释并修改 model_id：')
print("  # reg.upsert_entry({'id': 'your_model_id', 'stage': 'PRODUCTION'}, validate=False)")

# ── 手动晋级（取消注释即可）─────────────────
# PROMOTE_ID = 'us_absret_20260630'
# reg.upsert_entry({'id': PROMOTE_ID, 'stage': 'PRODUCTION'}, validate=False)
# print(f'✅ Promoted: {PROMOTE_ID} → PRODUCTION')

## 5. 流程终点确认

完成本 notebook 后，整个 pipeline 跑通：

```
00 下载+同步  ✅
01 因子研究   ✅
end_to_end   ✅  (训练 + 回测 + 注册)
02 信号验证   ✅
03 Spread研究 ✅
05 注册管理   ✅  ← 你在这里
```

如 IC 衰减提示 `⚠️ YES`，返回 `00` → `end_to_end` 重新训练新版本。